# Q&A across documents with LangChain and LangSmith

In [3]:

from langchain_community.document_loaders import WikipediaLoader, Docx2txtLoader, PyPDFLoader, TextLoader
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

In [2]:
import getpass
OPENAI_API_KEY = getpass.getpass('Enter your OPENAI_API_KEY')

## Setting up vector database and embeddings

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
embeddings_model = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
vector_db = Chroma('tourist_info', embeddings_model)

In [9]:
# wikipedia_loader = WikipediaLoader(query='Paestum')
# wikipedia_chunks = text_splitter.split_documents(wikipedia_loader.load())


In [10]:
word_loader = Docx2txtLoader('Paestum/Paestum-Britannica.docx')
word_chunks = text_splitter.split_documents(word_loader.load())
vector_db.add_documents(word_chunks)

['b96f7a1d-f814-4a92-8470-44a64642f10a',
 '8cb06860-6ee8-4fb1-99e0-1129c85c4caa',
 '2f449056-06b0-412a-9a6b-eeb5fa906726',
 '73d094f8-6ece-4611-b343-63ec2e2bda3c',
 '49841252-53ba-4fcf-b277-4d7c1a118e87',
 '5c0c508c-4ac1-48c0-ae29-0bf4ea6ca01f',
 '2b63096e-ce0e-481f-a8d1-7054d4eaa76b',
 '3c90898c-52d2-4e8f-ac44-7b926ab8dec2']

In [11]:
pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
pdf_chunks = text_splitter.split_documents(pdf_loader.load())
vector_db.add_documents(pdf_chunks)

['320a02e5-0f6a-4b6b-a9c8-6983642ecbfb',
 '74476cd4-01fc-4b38-9f6d-86057e3d92ad',
 'efb2b456-ead9-4af6-a9f4-7fd8b031cd93',
 '828e7fb8-b3ca-442a-801b-332cb195d682',
 '07d09e83-2efd-4b3f-a69c-99b446083781',
 'a6e57e33-9a01-4c82-8ef8-bfaad64e6695',
 '75f0c40d-e57c-4533-847f-8984121b3848',
 '2dac60de-8456-4a6f-afe6-6a34c0a64d3b',
 'cf03e739-c90d-40c3-a7e5-f33aac13c98f',
 'c216b522-7956-4e0f-aac8-3be491c88993',
 '85eb0c46-e34a-470d-8c85-a1124faeb5d4',
 '7e40f6d6-3ee4-4cc5-bf67-626c345d87ec',
 'e1578b67-c21a-4322-ae38-a4b3f73e9d8a',
 '12cfff12-09c4-489e-b734-bf9e10af9e85',
 '302ef0c6-b132-4e62-881f-0ab7d23739a2',
 '90863dce-7423-453f-9189-c74dfd4e0262',
 '233b3edc-521f-44b0-b1ba-0d3baaf57838',
 '3944678c-f55f-4f5d-a1bf-2dce8b2c5d07',
 'a1b3f285-1af1-489f-a5ad-8bd54a9947e7',
 '5b42f537-2b96-458b-addf-6499c225b5b3',
 'af8aaac3-32e0-4d45-acb6-c04ab81219d3',
 'a2669477-bc7c-413c-b7bb-ffbeb0a19da7',
 '8e888bd0-8d34-4b47-9829-d6008ed64291',
 '5b66e9bd-2c50-4672-870b-aa6609042bd8',
 'f7aaae07-91a5-

In [12]:
txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
txt_chunks = text_splitter.split_documents(txt_loader.load())
vector_db.add_documents(txt_chunks)

['a6fce8a9-e128-4a2e-9edf-1baf8125955a']

### Removing duplication

In [14]:
def split_and_import(loader):
     chunks = text_splitter.split_documents(loader.load())
     vector_db.add_documents(chunks)

In [15]:
# wikipedia_loader = WikipediaLoader(query="Paestum")
# split_and_import(wikipedia_loader)

word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
split_and_import(word_loader)

pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
split_and_import(pdf_loader)

txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
split_and_import(txt_loader)

## Ingesting Multiple Documents from a Folder (two techniques)

### 1) Iterating over all files in a folder

In [16]:
loader_classes = {
    'docx': Docx2txtLoader,
    'pdf': PyPDFLoader,
    'txt': TextLoader
}

In [17]:
import os
    
def get_loader(filename):
    _, file_extension = os.path.splitext(filename)
    file_extension = file_extension.lstrip('.')
    loader_class = loader_classes.get(file_extension)
    
    if loader_class:
        return loader_class(filename)
    
    else:
        raise ValueError(f"No loader available for file extension '{file_extension}'")

#### Ingesting the files from the folder (Exercise solution)

In [18]:
folder_path = "CilentoTouristInfo" #A Path to the folder containing the documents

for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    
    if os.path.isfile(file_path):
        try:
            loader = get_loader(file_path)
            split_and_import(loader)
        except ValueError as e:
            print(e)           

### 2) Ingesting all files with with DirectoryLoader

In [15]:
# ONLY RUN THIS IF YOU HAVE SUCCESFULLY INSTALLED unstructured or langchain-unstructured
# THE INSTALLATION IS OPERATIVE SYSTEM SPECIFIC
# follow LangChain instructions at https://python.langchain.com/v0.2/docs/integrations/providers/unstructured/ or 
# Unstructured instructions at https://docs.unstructured.io/welcome#quickstart-unstructured-open-source-library
folder_path = "CilentoTouristInfo"
pattern = "**/*.{docx,pdf,txt}" #A Pattern to match .docx, .pdf, and .txt files

directory_loader = DirectoryLoader(folder_path, pattern) #B Initialize the DirectoryLoader with the folder path and pattern
split_and_import(directory_loader)

NameError: name 'DirectoryLoader' is not defined

## Querying the vector store directly

In [19]:
query = "Where was Poseidonia and who renamed it to Paestum?" 
results = vector_db.similarity_search(query, 4)
print(results)

[Document(id='2f449056-06b0-412a-9a6b-eeb5fa906726', metadata={'source': 'Paestum/Paestum-Britannica.docx'}, page_content='Poseidonia was probably founded about 600\xa0BC\xa0by Greek colonists from\xa0Sybaris, along the\xa0Gulf of Taranto, and it had become a flourishing town by 540, judging from its temples. After many years’ resistance the city came under the domination of the\xa0Lucanians\xa0(an\xa0indigenous\xa0Italic people) sometime before 400\xa0BC, after which its name was changed to Paestum. Alexander, the king of Epirus, defeated the Lucanians at Paestum about 332\xa0BC, but the city remained Lucanian until 273, when it came under'), Document(id='5740b4ac-c858-4340-a232-aff1c1598853', metadata={'source': 'Paestum/Paestum-Britannica.docx'}, page_content='Poseidonia was probably founded about 600\xa0BC\xa0by Greek colonists from\xa0Sybaris, along the\xa0Gulf of Taranto, and it had become a flourishing town by 540, judging from its temples. After many years’ resistance the city 

In [20]:
len(results)

4

## Asking a question through a RAG chain

In [21]:
from openai import OpenAI
# import getpass

# OPENAI_API_KEY = getpass.getpass('Enter your OPENAI_API_KEY')

In [22]:
from langchain_core.prompts import PromptTemplate

rag_prompt_template = """Use the following pieces of context
to answer the question at the end. 
If you don't know the answer, just say that you don't know, 
don't try to make up an answer.
Use three sentences maximum and keep the 
answer as concise as possible.
{context}
Question: {question}
Helpful Answer:"""

rag_prompt = PromptTemplate.from_template(rag_prompt_template)

In [23]:
retriever = vector_db.as_retriever()

In [24]:
from langchain_core.runnables import RunnablePassthrough
question_feeder = RunnablePassthrough()

In [26]:
from langchain_openai import ChatOpenAI

chatbot = ChatOpenAI(openai_api_key=OPENAI_API_KEY, model_name='gpt-5-nano')

In [30]:
# set up RAG chain

rag_chain = {
    'context': retriever,
    'question': question_feeder
} | rag_prompt  | chatbot


In [28]:
def execute_chain(chain, question):
    answer = chain.invoke(question)
    return answer

In [31]:
question = """
Where was Poseidonia and who renamed it to Paestum.
Also tell me the source.
"""
answer = execute_chain(rag_chain, question)
print(answer.content)

- Poseidonia was located along the Gulf of Taranto in southern Italy.  
- It was renamed Paestum after coming under Lucanian domination, sometime before 400 BC.  
- Source: Britannica article in Paestum-Britannica.docx.


In [23]:
print(answer)

content='- Poseidonia was located along the Gulf of Taranto in southern Italy. \n- It was renamed Paestum by the Lucanians (before 400 BCE). \n- Source: Paestum-Britannica.docx (Britannica entry on Paestum).' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 1730, 'prompt_tokens': 1531, 'total_tokens': 3261, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1664, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CXXc1jjfRjBIebjlA9PoYPOMtbciD', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--ff3aa984-36d5-4f31-ad8e-d4ac12bdc6d8-0' usage_metadata={'input_tokens': 1531, 'output_tokens': 1730, 'total_tokens': 3261, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 

In [32]:
question = """
And then, what they do? 
Tell me only if you know. 
Also tell me the source.
""" 
answer = execute_chain(rag_chain, question)
print(answer.content)

The Way of Truth presents what is true; The Way of Opinion concerns what is believed or opinion. The Way of Truth is largely reconstructed from Sextus Empiricus and Simplicius, while The Way of Opinion survives only in small fragments. Source: CilentoTouristInfo\Parmenides.docx.


## Chatbot memory of message history

In [40]:
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnableLambda

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant, world-class expert in Roman and Greek history, especially in towns located in southern Italy. Provide interesting insights on local history and recommend places to visit with knowledgeable and engaging answers. Answer all questions to the best of your ability, but only use what has been provided in the context. If you don't know, just say you don't know. Use three sentences maximum and keep the answer as concise as possible."),
        ("placeholder", "{chat_history_messages}"),
        ("assistant", "{retrieved_context}"),
        ("human", "{question}"),
    ]
)

retriever = vector_db.as_retriever()
question_feeder = RunnablePassthrough()
chatbot = ChatOpenAI(openai_api_key=OPENAI_API_KEY, model_name="gpt-5-nano")
chat_history_memory = ChatMessageHistory()

rag_chain = {
    "retrieved_context": retriever, 
    "question": question_feeder,
    "chat_history_messages": RunnableLambda(lambda x: chat_history_memory.messages)
} | rag_prompt | chatbot

def execute_chain_with_memory(chain, question):
    chat_history_memory.add_user_message(question)
    answer = chain.invoke(question)
    chat_history_memory.add_ai_message(answer)
    print(f'Full chat message history: {chat_history_memory.messages}\n\n')                                      
    return answer

In [39]:
question = """Where was Poseidonia and who renamed 
it to Paestum? Also tell me the source."""
answer = execute_chain_with_memory(rag_chain, question)
print(answer.content)

Full chat message history: [HumanMessage(content='Where was Poseidonia and who renamed \nit to Paestum? Also tell me the source.', additional_kwargs={}, response_metadata={}), AIMessage(content='Poseidonia lay on the Gulf of Taranto in southern Italy and was founded around 600 BC by Greek colonists from Sybaris; its name was changed to Paestum after Lucanian domination before 400 BC. The exact timing and who renamed it are uncertain, though some scholars link the change to Lucanian rule. Sources: Britannica (Paestum-Britannica.docx) and PaestumRevisited.pdf.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1571, 'prompt_tokens': 930, 'total_tokens': 2501, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1472, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_finger

In [41]:
question = """And then what did they do? 
Also tell me the source""" 
answer = execute_chain_with_memory(rag_chain, question)
print(answer.content)

Full chat message history: [HumanMessage(content='And then what did they do? \nAlso tell me the source', additional_kwargs={}, response_metadata={}), AIMessage(content='I don’t have enough information in the provided excerpts to say what they did.  \nThe excerpts mention The Way of Truth, On Nature, and The Way of Opinion from CilentoTouristInfo\\Parmenides.docx and CilentoTouristInfo\\Cilento.docx.  \nSource: CilentoTouristInfo\\Parmenides.docx and CilentoTouristInfo\\Cilento.docx.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1374, 'prompt_tokens': 362, 'total_tokens': 1736, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1280, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DrkWtJN9ybERKhq3wOJ9xLdgYAYMl', 'service_tier':

## Tracing with LangSmith

Stop the notebook and open a new operative system shell (for example Windows command shell).

Configure the relevant environment variables in the OS shell, the rerun the previous Jupyter cells:
```bash
(env_ch07) C:\...\ch07>set LANGSMITH_TRACING=true
(env_ch07) C:\...\ch07>set LANGSMITH_ENDPOINT=https://api.smith.langchain.com
(env_ch07) C:\...\ch07>set LANGSMITH_PROJECT=Q & A chatbot
(env_ch07) C:\...\ch07>set LANGSMITH_API_KEY=<YOUR_LANGSMITH_API_KEY>
```

Then Restart the Jupyter notebook:
```bash
(env_ch07) C:\...\ch07>jupyter notebook 07-QA_across_documents.ipynb
```

Finally re-execute the whole Jupyter notebook cell by cell. All the activity will have not been logged through LangSmith.